In [1]:
# ===============================
# 1. Import libraries
# ===============================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet', quiet=True)

import warnings
warnings.filterwarnings('ignore')

SEED = 42

print("All libraries loaded successfully ✅")

/Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded successfully ✅


In [2]:

# Project root (notebook is inside notebooks/)
PROJECT_ROOT = Path("..").resolve()

# Input files
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# Output files
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SUBMISSION_PATH_050 = OUTPUT_DIR / "submission_threshold_050.csv"
SUBMISSION_PATH_072 = OUTPUT_DIR / "submission_threshold_072.csv"

# Load train data
df = pd.read_csv(TRAIN_PATH)

print("Train file:", TRAIN_PATH)
print("Test file:", TEST_PATH)
print("Output dir:", OUTPUT_DIR)

Train file: /Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/data/train.csv
Test file: /Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/data/test.csv
Output dir: /Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/outputs


In [3]:
#loading the train.csv dataset

df = pd.read_csv("../data/train.csv")

df.head()

,Id,livestream_id,username,text,date,time,channelname,subscriber_bi,IS_TOXIC
0,12,1,admirallove,"""""jan 6th was a peaceful protest""""",2023-08-02,2023-08-02 18:47:46+00:00,sondsol,1.0,0
1,13,1,i8urmom2,"unprecedented? it was a """"mostly peaceful"""" riot.",2023-08-02,2023-08-02 18:47:53+00:00,sondsol,0.0,0
2,14,1,i8urmom2,"unprecedented riots are those left wing """"prot...",2023-08-02,2023-08-02 18:48:15+00:00,sondsol,0.0,0
3,15,1,i8urmom2,"@AdmiralLove it was, by far. far far far far f...",2023-08-02,2023-08-02 18:48:44+00:00,sondsol,0.0,0
4,16,1,internet_gas,The prosecutor was between 2015 to 2016 Zloche...,2023-08-02,2023-08-02 18:48:45+00:00,sondsol,0.0,0


In [4]:
#Parsing date and time
df["time"] = pd.to_datetime(df["time"], errors="coerce")


df = df.sort_values("time").reset_index(drop=True)

print(df["time"].iloc[0], "to", df["time"].iloc[-1])

df.head()



2023-08-02 18:45:24+00:00 to 2023-09-11 23:36:07+00:00


,Id,livestream_id,username,text,date,time,channelname,subscriber_bi,IS_TOXIC
0,439,2,koshkathemerc,Ahhhhhh looks like I'm not collecting my bet t...,2023-08-02,2023-08-02 18:45:24+00:00,tyt,1.0,0
1,443,2,eurylino,She's the most disappointing Kraken of all tim...,2023-08-02,2023-08-02 18:45:24+00:00,tyt,1.0,0
2,440,2,hoosierdaddy89,Sickening Six,2023-08-02,2023-08-02 18:45:24+00:00,tyt,0.0,0
3,441,2,hoosierdaddy89,Like the Chicago 7,2023-08-02,2023-08-02 18:45:24+00:00,tyt,0.0,0
4,442,2,hoosierdaddy89,need a name,2023-08-02,2023-08-02 18:45:24+00:00,tyt,0.0,0


## Train/Validation Split
Splitting the data right now **before any feature engineering**, using 90/10 split. All feature engineering will then be fitted on the training set only, and applied to the validation and test set to avoid any form of **data leakage** where information from the test set influences how we build features. Because the data is ordered by time, we take the first 90% as training, the next 10% as validation. 


We split the data using `GroupShuffleSplit` based on `livestream_id`. This ensures that all comments from the same livestream session are assigned entirely to either the training set or the validation set, reducing leakage across splits.

This is important because comments within the same livestream can share common context, style, and discussion topics. A grouped split therefore gives a more realistic estimate of generalisation to unseen livestream sessions.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(test_size=0.1, random_state=42)

train_indices, val_indices = next(
    splitter.split(df, groups=df["livestream_id"])
)

train_data = df.iloc[train_indices].copy()
validation_data = df.iloc[val_indices].copy()

y_train = train_data["IS_TOXIC"].values
y_validation = validation_data["IS_TOXIC"].values

print(f"Training samples:   {len(train_data):,}")
print(f"Validation samples: {len(validation_data):,}")
print()
print(f"Train date range: {train_data['time'].iloc[0].date()} to {train_data['time'].iloc[-1].date()}")
print(f"Val date range:   {validation_data['time'].iloc[0].date()} to {validation_data['time'].iloc[-1].date()}")

Training samples:   106,254
Validation samples: 13,563

Train date range: 2023-08-02 to 2023-09-11
Val date range:   2023-08-02 to 2023-09-06


### Using the package BERT


Prepare train and validation datasets for BERT

In [9]:
# Keep only the text and label columns for BERT
train_small = train_data[["text", "IS_TOXIC"]].copy()
validation_small = validation_data[["text", "IS_TOXIC"]].copy()

# Rename target column to 'label' because Hugging Face expects that
train_small = train_small.rename(columns={"IS_TOXIC": "label"})
validation_small = validation_small.rename(columns={"IS_TOXIC": "label"})

train_small.head()


,text,label
0,Ahhhhhh looks like I'm not collecting my bet t...,0
1,She's the most disappointing Kraken of all tim...,0
2,Sickening Six,0
3,Like the Chicago 7,0
4,need a name,0


In [8]:
validation_small.head()

,text,label
38,"""""jan 6th was a peaceful protest""""",0
39,"unprecedented? it was a """"mostly peaceful"""" riot.",0
46,"unprecedented riots are those left wing """"prot...",0
52,"@AdmiralLove it was, by far. far far far far f...",0
53,The prosecutor was between 2015 to 2016 Zloche...,0


Convert pandas dataframes to Hugging Face datasets

In [10]:
hf_train = Dataset.from_pandas(train_small.reset_index(drop=True))
hf_val = Dataset.from_pandas(validation_small.reset_index(drop=True))

print(hf_train)
print(hf_val)

Dataset({
    features: ['text', 'label'],
    num_rows: 106254
})
Dataset({
    features: ['text', 'label'],
    num_rows: 13563
})


Load Tokenizer

Start with DistilBERT. It is lighter than full BERT and easier to train.

In [11]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Tokenize the text

In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_val = hf_val.map(tokenize_function, batched=True)

Map: 100%|██████████| 13563/13563 [00:00<00:00, 49800.97 examples/s]


Metrics

In [13]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    precision = precision_metric.compute(predictions=preds, references=labels)
    recall = recall_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels)

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"],
    }

Load the model

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6175.90it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#training settings

In [19]:


training_args = TrainingArguments(
    output_dir="../outputs/bert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=False
)

Trainer

In [20]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Train BERT

In [21]:
trainer.train()

RuntimeError: MPS backend out of memory (MPS allocated: 2.23 GiB, other allocations: 6.77 GiB, max allowed: 9.07 GiB). Tried to allocate 89.42 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).